# Tokenization Techniques from Scratch

### 🔑 Interview Comparison Summary

| Tokenizer  | Used in       | Idea             |
| ---------- | ------------- | ---------------- |
| Whitespace | classical NLP | split by space   |
| Char       | early seq2seq | characters       |
| BPE        | GPT, RoBERTa  | frequent merge   |
| WordPiece  | BERT          | likelihood merge |
| Unigram    | SentencePiece | prune vocab      |
| Byte-BPE   | GPT-2, LLaMA  | bytes + BPE      |


### 1️⃣ Simple Tokenization (Whitespace / Rule-based)

#### Concept

Split text into tokens using delimiters (space, punctuation).

👉 Old NLP pipelines (before subword models)
👉 Still used for preprocessing / baseline models

### Implementation — whitespace tokenizer

In [1]:
def whitespace_tokenize(text: str):
    return text.strip().split()

text = "Tokenization is the first step in NLP."
print(whitespace_tokenize(text))

['Tokenization', 'is', 'the', 'first', 'step', 'in', 'NLP.']


Output:

```
['Tokenization', 'is', 'the', 'first', 'step', 'in', 'NLP.']
```

In [ ]:
## Improved: punctuation-aware tokenizer

import re

def simple_tokenize(text: str):
    # split words and punctuation separately
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

text = "Tokenization is the first step in NLP."
print(simple_tokenize(text))


Output:

```
['Tokenization', 'is', 'the', 'first', 'step', 'in', 'NLP', '.']
```

# 2️⃣ Character Tokenization

## Concept

Split text into characters.

👉 Useful for:

* morphologically rich languages
* unknown word handling
* early seq2seq models

In [ ]:
def char_tokenize(text):
    return list(text)

print(char_tokenize("NLP"))

Output:

```
['N', 'L', 'P']
```

# 3️⃣ N-gram Tokenization

## Concept

Token = sequence of N characters or words.

Examples:

* unigram: "I"
* bigram: "I love"
* trigram: "I love NLP"

---

## Word n-grams

In [ ]:
def word_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

tokens = simple_tokenize("I love NLP tokenization")
print(word_ngrams(tokens, 2))

Output:

```
[('I', 'love'), ('love', 'NLP'), ('NLP', 'tokenization')]
```


## Character n-grams

In [ ]:
def char_ngrams(text, n):
    return [text[i:i+n] for i in range(len(text)-n+1)]

print(char_ngrams("token", 3))

Output:

```
['tok', 'oke', 'ken']
```


# 4️⃣ Byte Pair Encoding (BPE)

## Concept

Most important for interviews.

Idea:

1. Start with characters
2. Find most frequent pair
3. Merge pair
4. Repeat

Example:

```
low
lowest
lower
```

Frequent pairs:

```
lo → merge
low → merge
```


In [ ]:
# BPE From Scratch (Train)

from collections import Counter, defaultdict

def get_vocab(words):
    vocab = Counter()
    for word in words:
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] += 1
    return vocab

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

In [ ]:
## Train BPE merges

words = ["low", "lower", "lowest", "newest", "widest"]

vocab = get_vocab(words)

num_merges = 10
merges = []

for _ in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    merges.append(best)
    vocab = merge_vocab(best, vocab)

print("Learned merges:", merges)

In [ ]:
## Apply BPE to tokenize new word

def apply_bpe(word, merges):
    tokens = list(word) + ["</w>"]
    for pair in merges:
        i = 0
        while i < len(tokens)-1:
            if (tokens[i], tokens[i+1]) == pair:
                tokens[i:i+2] = ["".join(pair)]
            else:
                i += 1
    return tokens

print(apply_bpe("lowest", merges))


# 5️⃣ WordPiece (BERT Tokenizer)

## Concept

Similar to BPE but:

Key difference:

👉 BPE → most frequent pair
👉 WordPiece → best likelihood improvement

Also uses prefix marker:

```
playing → play ##ing
```

---

# WordPiece Training (Simplified)

We approximate likelihood by frequency ratio.


In [ ]:
from collections import Counter

def wordpiece_train(words, vocab_size=50):
    vocab = set()
    corpus = []

    for word in words:
        chars = list(word)
        corpus.append(chars)
        vocab.update(chars)

    vocab = set(vocab)

    def get_pair_scores(corpus):
        pair_freq = Counter()
        single_freq = Counter()

        for tokens in corpus:
            for t in tokens:
                single_freq[t] += 1
            for i in range(len(tokens)-1):
                pair_freq[(tokens[i], tokens[i+1])] += 1

        scores = {}
        for pair, freq in pair_freq.items():
            scores[pair] = freq / (single_freq[pair[0]] * single_freq[pair[1]])
        return scores

    while len(vocab) < vocab_size:
        scores = get_pair_scores(corpus)
        if not scores:
            break
        best = max(scores, key=scores.get)

        new_token = best[0] + best[1]
        vocab.add(new_token)

        # merge in corpus
        new_corpus = []
        for tokens in corpus:
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                    new_tokens.append(new_token)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_corpus.append(new_tokens)
        corpus = new_corpus

    return vocab

In [ ]:
## Test WordPiece

words = ["playing", "played", "player", "plays"]
vocab = wordpiece_train(words, vocab_size=30)

print(vocab)

# 6️⃣ Unigram LM Tokenization (SentencePiece)

## Concept

Opposite of BPE.

BPE:

* start small → merge

Unigram:

* start large vocab
* remove tokens that hurt likelihood least

We simulate simplified version.



In [ ]:
import random

def unigram_tokenize(word, vocab):
    tokens = []
    i = 0
    while i < len(word):
        matched = None
        for j in range(len(word), i, -1):
            sub = word[i:j]
            if sub in vocab:
                matched = sub
                break
        if matched is None:
            matched = word[i]
        tokens.append(matched)
        i += len(matched)
    return tokens

vocab = {"play", "ing", "er", "ed", "pla", "y"}
print(unigram_tokenize("player", vocab))

# 7️⃣ Byte-Level BPE (GPT-2 / LLaMA)

## Concept

Instead of characters → use UTF-8 bytes.

Advantages:

* works for ANY language
* no unknown token
* emojis supported

Example:

```
"é" → bytes: [195, 169]
```

## Byte tokenizer demo

In [ ]:
def byte_tokenize(text):
    return list(text.encode("utf-8"))

print(byte_tokenize("Hello"))
print(byte_tokenize("é"))

Output:

```
[72, 101, 108, 108, 111]
[195, 169]
```


Then BPE merges operate on bytes instead of chars.

### 🎯 Interview Questions You May Get

**Q1:** Why subword tokenization?
👉 Handle rare/unknown words

**Q2:** BPE vs WordPiece?
👉 freq vs likelihood

**Q3:** Why byte-level?
👉 multilingual + no OOV

**Q4:** Why not word tokenizer?
👉 huge vocab, OOV problem

